#### Base Line Experiment Setup
- test pipeline performance with the base classifier on each dataset individually

In [1]:
# imports
import torch
from torch.utils.data import DataLoader
from dataset_utils import *
from ClassificationHeads import BaselineClassificationHead
from sklearn.preprocessing import RobustScaler
import umap
from transformers import DistilBertTokenizer, AutoModel
from pipeline_utils import create_profile_vector, create_tweet_vectors, test_classifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from pipeline_utils import train_classifier
import numpy as np
import random
device = 'cuda' if torch.cuda.is_available() else 'cpu'

/home/max/anaconda3/envs/social-media-bot-detection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def train_loop(data_loader, tokenizer, embedding_model, dim_reducer, scaler, classifier):
    ground_truth_labels = []
    profile_vectors = []
    tweet_vectors = []
    for i, sample in enumerate(data_loader):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, embedding_model, max_tweets= 50, batch_size = 50)

        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

        # append feature vectors and ground truth
        profile_vectors.append(profile_embed)
        tweet_vectors.append(tweet_vec)
        ground_truth_labels.append(sample.label)
        print(f"\rIteration: {i}", end="")

    tweet_vectors = np.array(tweet_vectors)
    ground_truth_labels = np.array(ground_truth_labels)
    profile_vectors = np.array(profile_vectors)

    # reduce tweet vectors in dimensionality
    if(len(tweet_vectors) > 100000):
        indices = random.sample(range(len(tweet_vectors)), 100000)
        fitting_vectors = tweet_vectors[indices]
        fitting_labels = ground_truth_labels[indices]
    else:
        fitting_vectors = tweet_vectors
        fitting_labels = ground_truth_labels

    dim_reducer.fit(X=fitting_vectors, y=torch.tensor(fitting_labels))
    reduced_tweet_vectors = dim_reducer.transform(tweet_vectors)
    reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)

    # scale profile vectors
    scaled_profile_vectors = scaler.fit_transform(profile_vectors)

    # combine feature vectors
    embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, scaled_profile_vectors)]

    # perform classification training
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
    ground_truths, predictions = train_classifier(classifier, criterion, optimizer, embedding_features, ground_truth_labels)

    # performance
    print("[training performance]")
    print("accuracy: ", accuracy_score(ground_truths, predictions))
    print("recall: ", recall_score(ground_truths, predictions, average="macro"))
    print("f1 score: ", f1_score(ground_truths, predictions, average="macro"))
    print(confusion_matrix(ground_truths, predictions))
    return predictions, ground_truths

def testing_loop(data_loader, tokenizer, embedding_model, dim_reducer, scaler, classifier):
    ground_truth_labels = []
    profile_vectors = []
    tweet_vectors = []
    for i, sample in enumerate(data_loader):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, embedding_model, max_tweets= 50, batch_size = 50)

        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

        # append feature vectors and ground truth
        profile_vectors.append(profile_embed)
        tweet_vectors.append(tweet_vec)
        ground_truth_labels.append(sample.label)
        print(f"\rIteration: {i}", end="")

    tweet_vectors = np.array(tweet_vectors)
    ground_truth_labels = np.array(ground_truth_labels)
    profile_vectors = np.array(profile_vectors)

    # reduce tweet vectors in dimensionality
    if(len(tweet_vectors) > 100000):
        indices = random.sample(range(len(tweet_vectors)), 100000)
        fitting_vectors = tweet_vectors[indices]
        fitting_labels = ground_truth_labels[indices]
    else:
        fitting_vectors = tweet_vectors
        fitting_labels = ground_truth_labels

    dim_reducer.fit(X=fitting_vectors, y=torch.tensor(fitting_labels))
    reduced_tweet_vectors = dim_reducer.transform(tweet_vectors)
    reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)

    # scale profile vectors
    scaled_profile_vectors = scaler.fit_transform(profile_vectors)

    # combine feature vectors
    embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, scaled_profile_vectors)]

    # perform classification training
    ground_truths, predictions = test_classifier(classifier, embedding_features, ground_truth_labels)

    # performance
    print("[testing performance]")
    print("accuracy: ", accuracy_score(ground_truths, predictions))
    print("recall: ", recall_score(ground_truths, predictions, average="macro"))
    print("f1 score: ", f1_score(ground_truths, predictions, average="macro"))
    print(confusion_matrix(ground_truths, predictions))
    return predictions, ground_truths

In [3]:
# Dataset: Cresci-2017

# init relevant models
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
dim_reducer = umap.UMAP(n_components=16, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
scaler = RobustScaler(with_centering=False)
classifier = BaselineClassificationHead(23,6).to(device)

# train loop
dataset_mode = "train"
user_dataset = Cresci17(Cresci17SetTypes.GENUINE_USER, dataset_mode, root="./datasets", custom_label=0)
fake_dataset = Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, dataset_mode, root="./datasets", custom_label=1)
social1_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, dataset_mode, root="./datasets", custom_label=2)
social2_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, dataset_mode, root="./datasets", custom_label=3)
social3_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, dataset_mode, root="./datasets", custom_label=4)
traditional1_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, dataset_mode, root="./datasets", custom_label=5)
combo_dataset = InterleavedIterableDataset([user_dataset, fake_dataset, social1_dataset, social2_dataset, social3_dataset, traditional1_dataset], "RoundRobin")
dataset = DataLoader(combo_dataset, batch_size=None,batch_sampler=None, pin_memory=True)
train_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)

# test loop
dataset_mode = "test"
user_dataset = Cresci17(Cresci17SetTypes.GENUINE_USER, dataset_mode, root="./datasets", custom_label=0)
fake_dataset = Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, dataset_mode, root="./datasets", custom_label=1)
social1_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, dataset_mode, root="./datasets", custom_label=2)
social2_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, dataset_mode, root="./datasets", custom_label=3)
social3_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, dataset_mode, root="./datasets", custom_label=4)
traditional1_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, dataset_mode, root="./datasets", custom_label=5)
combo_dataset = InterleavedIterableDataset([user_dataset, fake_dataset, social1_dataset, social2_dataset, social3_dataset, traditional1_dataset], "RoundRobin")
dataset = DataLoader(combo_dataset, batch_size=None,batch_sampler=None, pin_memory=True)
testing_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)
print("--done--")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 21296.29it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 8886Epoch [1/15] - Loss: 0.4582
Epoch [2/15] - Loss: 0.1692
Epoch [3/15] - Loss: 0.1502
Epoch [4/15] - Loss: 0.1430
Epoch [5/15] - Loss: 0.1418
Epoch [6/15] - Loss: 0.1317
Epoch [7/15] - Loss: 0.1380
Epoch [8/15] - Loss: 0.1309
Epoch [9/15] - Loss: 0.1368
Epoch [10/15] - Loss: 0.1321
Epoch [11/15] - Loss: 0.1357
Epoch [12/15] - Loss: 0.1491
Epoch [13/15] - Loss: 0.1403
Epoch [14/15] - Loss: 0.1403
Epoch [15/15] - Loss: 0.1300
[training performance]
accuracy:  0.9632046809947113
recall:  0.9251858051783982
f1 score:  0.9351471311763738
[[2510   32    4    3    5    9]
 [  47 2240    0   12    3   46]
 [   5    2   60    3    0    3]
 [   9   27    1 2725    0    2]
 [  12    3    1    5  327    8]
 [  13   52    1   15    4  698]]
Iteration: 1146[testing performance]
accuracy:  0.3792502179598954
recall:  0.4798424362822851
f1 score:  0.3465511858528412
[[ 99   0   0 217   0   7]
 [ 14 222  30   4   0  11]
 [  1   1  11   0   0   0]
 [  2 355   0   0   4   4]
 [  3   1   0   

In [4]:
# Dataset: Cresci 2018

# init relevant models
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
dim_reducer = umap.UMAP(n_components=16, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
scaler = RobustScaler(with_centering=False)
classifier = BaselineClassificationHead(23,3).to(device)

# training loop
dataset = DataLoader(Cresci18("train", root="./datasets", use_only_labelled=True, label_mapping=[0,1,2]), batch_size=None,batch_sampler=None, pin_memory=True)
train_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)

# testing loop
dataset = DataLoader(Cresci18("test", root="./datasets", use_only_labelled=True, label_mapping=[0,1,2]), batch_size=None,batch_sampler=None, pin_memory=True)
testing_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)
print("--done--")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9606.30it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 20834Epoch [1/15] - Loss: 0.5933
Epoch [2/15] - Loss: 0.4222
Epoch [3/15] - Loss: 0.4283
Epoch [4/15] - Loss: 0.4258
Epoch [5/15] - Loss: 0.4207
Epoch [6/15] - Loss: 0.4239
Epoch [7/15] - Loss: 0.4200
Epoch [8/15] - Loss: 0.4193
Epoch [9/15] - Loss: 0.4138
Epoch [10/15] - Loss: 0.4153
Epoch [11/15] - Loss: 0.4214
Epoch [12/15] - Loss: 0.4291
Epoch [13/15] - Loss: 0.4189
Epoch [14/15] - Loss: 0.4159
Epoch [15/15] - Loss: 0.4101
[training performance]
accuracy:  0.8160307175425966
recall:  0.7439910601032795
f1 score:  0.7593729004877817
[[13556  1275]
 [ 2558  3446]]
Iteration: 2593[testing performance]
accuracy:  0.5967617579028527
recall:  0.6519651001038897
f1 score:  0.5865330401872981
[[978 884]
 [162 570]]
--done--


In [3]:
# Dataset Caverlee 11

# init relevant models
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
dim_reducer = umap.UMAP(n_components=16, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
scaler = RobustScaler(with_centering=False)
classifier = BaselineClassificationHead(23,3).to(device)

# training loop
dataset = DataLoader(Caverlee11("train", root="./datasets", label_mapping=[0,1]), batch_size=None,batch_sampler=None, pin_memory=True)
train_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)

# testing loop
dataset = DataLoader(Caverlee11("test", root="./datasets", label_mapping=[0,1]), batch_size=None,batch_sampler=None, pin_memory=True)
testing_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)
print("--done--")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8548.64it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 31943Epoch [1/15] - Loss: 0.3056
Epoch [2/15] - Loss: 0.2379
Epoch [3/15] - Loss: 0.2364
Epoch [4/15] - Loss: 0.2385
Epoch [5/15] - Loss: 0.2373
Epoch [6/15] - Loss: 0.2425
Epoch [7/15] - Loss: 0.2375
Epoch [8/15] - Loss: 0.2438
Epoch [9/15] - Loss: 0.2397
Epoch [10/15] - Loss: 0.2355
Epoch [11/15] - Loss: 0.2407
Epoch [12/15] - Loss: 0.2371
Epoch [13/15] - Loss: 0.2403
Epoch [14/15] - Loss: 0.2394
Epoch [15/15] - Loss: 0.2406
[training performance]
accuracy:  0.9305972952667168
recall:  0.9306122095341367
f1 score:  0.930496745955574
[[15471  1160]
 [ 1057 14256]]
Iteration: 3948[testing performance]
accuracy:  0.9731577614585971
recall:  0.9732131864072888
f1 score:  0.9731565066072109
[[1908   76]
 [  30 1935]]
--done--


In [3]:
# Dataset: Twibot 20
train_dataset = Twibot20("train", root="./datasets", label_mapping=[0,1])

# init relevant models
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
dim_reducer = umap.UMAP(n_components=16, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
scaler = RobustScaler(with_centering=False)
classifier = BaselineClassificationHead(23,3).to(device)

# training loop
dataset = DataLoader(Twibot20("train", root="./datasets", label_mapping=[0,1]), batch_size=None,batch_sampler=None, pin_memory=True)
train_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)

# testing loop
dataset = DataLoader(Twibot20("test", root="./datasets", label_mapping=[0,1]), batch_size=None,batch_sampler=None, pin_memory=True)
testing_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)
print("--done--")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9177.31it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 9504Epoch [1/15] - Loss: 0.2826
Epoch [2/15] - Loss: 0.2384
Epoch [3/15] - Loss: 0.2396
Epoch [4/15] - Loss: 0.2381
Epoch [5/15] - Loss: 0.2420
Epoch [6/15] - Loss: 0.2315
Epoch [7/15] - Loss: 0.2419
Epoch [8/15] - Loss: 0.2420
Epoch [9/15] - Loss: 0.2405
Epoch [10/15] - Loss: 0.2503
Epoch [11/15] - Loss: 0.2596
Epoch [12/15] - Loss: 0.2435
Epoch [13/15] - Loss: 0.2377
Epoch [14/15] - Loss: 0.2383
Epoch [15/15] - Loss: 0.2466
[training performance]
accuracy:  0.933929510783798
recall:  0.9336647876594353
f1 score:  0.9331022214427938
[[4967  340]
 [ 288 3910]]
Iteration: 1148[testing performance]
accuracy:  0.44386422976501305
recall:  0.5
f1 score:  0.3074141048824593
[[  0 639]
 [  0 510]]
--done--


In [3]:
# Dataset: Twibot 22

# init relevant models
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
dim_reducer = umap.UMAP(n_components=16, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25, low_memory=True)
scaler = RobustScaler(with_centering=False)
classifier = BaselineClassificationHead(23,3).to(device)

# training loop
dataset = DataLoader(Twibot22("train", root="./datasets", label_mapping=[0,1]), batch_size=None,batch_sampler=None, pin_memory=True)
train_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)

# testing loop
dataset = DataLoader(Twibot22("test", root="./datasets", label_mapping=[0,1]), batch_size=None,batch_sampler=None, pin_memory=True)
testing_loop(dataset, tokenizer, model, dim_reducer, scaler, classifier)
print("--done--")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8994.67it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--> Loading labels...
--> Loaded 800150 labels for split 'train'
--> Loading users (This will take a while via ijson)...
    Processed 50000 users...
--> Successfully cached 50000 users in memory.
    Processed 100000 users...
--> Successfully cached 100000 users in memory.
    Processed 150000 users...
--> Successfully cached 150000 users in memory.
    Processed 200000 users...
--> Successfully cached 200000 users in memory.
    Processed 250000 users...
--> Successfully cached 250000 users in memory.
    Processed 300000 users...
--> Successfully cached 300000 users in memory.
    Processed 350000 users...
--> Successfully cached 350000 users in memory.
    Processed 400000 users...
--> Successfully cached 400000 users in memory.
    Processed 450000 users...
--> Successfully cached 450000 users in memory.
    Processed 500000 users...
--> Successfully cached 500000 users in memory.
    Processed 550000 users...
--> Successfully cached 550000 users in memory.
    Processed 600000 us